# Notebook 05 — Multi-head attention

Paired with `theory/05_multi_head.md`. **Mode:** mixed.

## QHMPC

**Question.** On a task with two genuinely independent retrieval channels — does multi-head attention *spontaneously* split the work, with different heads handling different channels?

Specifically:
1. Structural: param/FLOP invariance vs. $h$.
2. Rank: pre-softmax score matrix is rank $\le d_h$, post-softmax is generically full rank.
3. **Head specialization on a two-channel task** — the load-bearing experiment.
4. Single-head baseline at matched parameters: does it solve the two-channel task at all? At what cost?
5. Per-head ablation: which heads matter for which channel?

**Hypothesis.** A 1-head model is fundamentally bottlenecked on a two-channel task: it has to use one attention pattern per query position, but the two channels demand different attention patterns. A 2+head model can dedicate one head per channel and trivially win. With $h \ge 2$, expect clean specialization; with $h = 1$, expect either chance accuracy on one channel or a forced compromise on both.

**Method.** Two-channel retrieve task — the model sees a sequence with `(channel-flag, key, value)` triples and a query asking "the value associated with this key, on channel C." Two channels mean the model must learn two retrieval patterns. Train at $h \in \{1, 2, 4\}$, ablate per head, visualize attention.

**Prediction.** *Paste your §5.7 predictions.*

**Confidence.** *[low / medium / high, reason]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Experiment 1 — parameter invariance under $h$

In [ ]:
class MHA(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.d_model, self.h, self.d_h = d_model, h, d_model // h
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X, return_attn=False):
        B, n, d = X.shape
        Q = self.W_Q(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        K = self.W_K(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        V = self.W_V(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        A = (Q @ K.transpose(-1, -2) / (self.d_h ** 0.5)).softmax(-1)
        out = (A @ V).transpose(1, 2).contiguous().view(B, n, d)
        out = self.W_O(out)
        return (out, A) if return_attn else out

for h in [1, 2, 4, 8]:
    m = MHA(64, h)
    p = sum(p.numel() for p in m.parameters())
    print(f'h = {h}, d_h = {64//h}: params = {p}')

**Sub-finding 1.** *Param count invariant under $h$? Expected by Prop 5.3.*

## Experiment 2 — rank lifting by softmax (Exercise 5.3)

$d_h = 8, n = 32$: pre-softmax score matrix is rank $\le d_h = 8$; post-softmax should generically be full rank $n = 32$.

In [ ]:
n, d_h = 32, 8
Q = np.random.randn(n, d_h); K = np.random.randn(n, d_h)
S = Q @ K.T / np.sqrt(d_h)
A = np.exp(S - S.max(axis=-1, keepdims=True))
A /= A.sum(axis=-1, keepdims=True)
print(f'rank(S) = {np.linalg.matrix_rank(S, tol=1e-8)}  (predicted ≤ {d_h})')
print(f'rank(A) = {np.linalg.matrix_rank(A, tol=1e-8)}  (predicted = {n})')

**Sub-finding 2.** *Did softmax lift the rank? What nonlinearity is doing it — the exponential? What would happen with a polynomial replacement?*

---
## Experiment 3 — TWO-CHANNEL retrieval task (the chapter's experiment)

**Task design.** Vocabulary: 8 channel-A keys, 8 channel-A values, 8 channel-B keys, 8 channel-B values, plus a `Q` query token. Sequence layout: `[ka1, va1, kb1, vb1, ka2, va2, kb2, vb2, ka3, va3, kb3, vb3, Q, kq, c]`. The last three tokens are the query: `Q` flag, the queried key `kq`, and a channel flag `c ∈ {A, B}`. The model must output the value associated with `kq` *on the right channel*.

**Why this is two-channel.** Channel-A and channel-B are encoded as disjoint vocabulary regions. The matching key for the query lives at one of the channel-appropriate positions. The model must (a) attend to keys of the right channel, (b) match the query key. A single-head model has to encode both decisions in one attention pattern per query position, conflating them. Two heads can split the work: one routes by channel, one routes by key.

**Prediction.** $h = 1$ should hit a hard ceiling around 50% (gets channel right but conflates within-channel keys, or vice-versa). $h \ge 2$ should saturate near 100%.

In [ ]:
# Vocabulary layout
K_PER = 8
V_PER = 8
VOCAB = {
    'kA': list(range(0, K_PER)),                              # 0..7
    'vA': list(range(K_PER, K_PER + V_PER)),                  # 8..15
    'kB': list(range(K_PER + V_PER, 2 * K_PER + V_PER)),      # 16..23
    'vB': list(range(2 * K_PER + V_PER, 2 * (K_PER + V_PER))),# 24..31
    'Q':  2 * (K_PER + V_PER),                                # 32
    'flA': 2 * (K_PER + V_PER) + 1,                           # 33
    'flB': 2 * (K_PER + V_PER) + 2,                           # 34
}
V_TOTAL = VOCAB['flB'] + 1
N_PAIRS = 3              # 3 channel-A pairs and 3 channel-B pairs
SEQ_LEN = 4 * N_PAIRS + 3   # interleaved (kA,vA, kB,vB)*N + Q + queried_key + channel_flag
OUT_VOCAB = V_PER * 2     # output is a value (channel-A or channel-B)

def make_batch(B):
    seqs = torch.zeros(B, SEQ_LEN, dtype=torch.long)
    targets = torch.zeros(B, dtype=torch.long)
    for b in range(B):
        kA = np.random.permutation(VOCAB['kA'])[:N_PAIRS]
        vA = np.random.choice(VOCAB['vA'], size=N_PAIRS)
        kB = np.random.permutation(VOCAB['kB'])[:N_PAIRS]
        vB = np.random.choice(VOCAB['vB'], size=N_PAIRS)
        for i in range(N_PAIRS):
            seqs[b, 4*i] = int(kA[i])
            seqs[b, 4*i+1] = int(vA[i])
            seqs[b, 4*i+2] = int(kB[i])
            seqs[b, 4*i+3] = int(vB[i])
        # query: Q flag + chosen key + channel flag
        chan = np.random.choice(['A', 'B'])
        if chan == 'A':
            qi = np.random.randint(N_PAIRS)
            seqs[b, -3] = VOCAB['Q']; seqs[b, -2] = int(kA[qi]); seqs[b, -1] = VOCAB['flA']
            targets[b] = int(vA[qi]) - K_PER       # output index in [0, V_PER + V_PER)
        else:
            qi = np.random.randint(N_PAIRS)
            seqs[b, -3] = VOCAB['Q']; seqs[b, -2] = int(kB[qi]); seqs[b, -1] = VOCAB['flB']
            targets[b] = int(vB[qi]) - K_PER - V_PER + V_PER     # = vB - K_PER - V_PER + V_PER, in [V_PER, 2*V_PER)
    return seqs, targets

# sanity
s, t = make_batch(1)
print(f'seq: {s[0].tolist()}')
print(f'target: {t[0].item()}  (in [0, {OUT_VOCAB}))')

In [ ]:
class Model(nn.Module):
    def __init__(self, h, d_model=32):
        super().__init__()
        self.embed = nn.Embedding(V_TOTAL, d_model)
        self.pos = nn.Parameter(torch.randn(SEQ_LEN, d_model) * 0.02)
        self.mha = MHA(d_model, h)
        self.out = nn.Linear(d_model, OUT_VOCAB, bias=False)
    def forward(self, seq, return_attn=False):
        x = self.embed(seq) + self.pos
        if return_attn:
            y, A = self.mha(x, return_attn=True)
            return self.out(y[:, -1]), A
        return self.out(self.mha(x)[:, -1])

def train_eval(h, steps=3000):
    torch.manual_seed(0)
    m = Model(h)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    losses = []
    for step in range(steps):
        seq, tgt = make_batch(64)
        loss = F.cross_entropy(m(seq), tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    with torch.no_grad():
        seq, tgt = make_batch(2000)
        acc = (m(seq).argmax(-1) == tgt).float().mean().item()
    return m, losses, acc

results = {}
for h in [1, 2, 4]:
    m, losses, acc = train_eval(h)
    results[h] = (m, losses, acc)
    print(f'h = {h}: final acc = {acc:.3f}   (chance = 1/{OUT_VOCAB} = {1/OUT_VOCAB:.3f})')

fig, ax = plt.subplots(figsize=(7, 4))
for h in [1, 2, 4]:
    ax.plot(results[h][1], label=f'h = {h}')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.set_yscale('log')
ax.set_title('Two-channel retrieval — does h=1 hit a ceiling?')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 3.** *Did $h = 1$ hit a ceiling? At what accuracy? Did $h \ge 2$ saturate? This is the core experiment — if $h = 1$ matches $h = 2$, the two-channel framing is wrong and the chapter's claim is weaker than stated.*

## Experiment 4 — visualize per-head attention after training ($h = 2$)

In [ ]:
m = results[2][0]
m.eval()
# Show one channel-A query and one channel-B query side by side
seqs, tgts = make_batch(8)
with torch.no_grad():
    _, A = m(seqs, return_attn=True)
A = A.numpy()  # (B, h, n, n)

# Find one A query and one B query in the batch
for idx in range(len(seqs)):
    if seqs[idx, -1].item() == VOCAB['flA']:
        idxA = idx; break
for idx in range(len(seqs)):
    if seqs[idx, -1].item() == VOCAB['flB']:
        idxB = idx; break

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for col, (qi, label) in enumerate([(idxA, 'A query'), (idxB, 'B query')]):
    for row in range(2):
        im = axes[row, col].imshow(A[qi, row], cmap='viridis', vmin=0)
        axes[row, col].set_title(f'Head {row} on {label}')
        axes[row, col].set_xlabel('key pos'); axes[row, col].set_ylabel('query pos')
plt.tight_layout(); plt.show()

print('Query-row attention for both queries:')
print(f'A-query, head 0: {A[idxA, 0, -1].round(3)}')
print(f'A-query, head 1: {A[idxA, 1, -1].round(3)}')
print(f'B-query, head 0: {A[idxB, 0, -1].round(3)}')
print(f'B-query, head 1: {A[idxB, 1, -1].round(3)}')

**Sub-finding 4.** *On the A-query, is the attention concentrated on channel-A positions (kA, vA at indices 0,1,4,5,8,9) or channel-B positions (2,3,6,7,10,11)? Does it differ between A and B queries? Did head 0 specialize to one channel and head 1 to the other?*

## Experiment 5 — per-head ablation by channel

If heads specialized, ablating one should hurt one channel selectively.

In [ ]:
import types

def eval_with_mask(model, head_mask):
    def patched(self, X, return_attn=False):
        B, n, d = X.shape
        Q = self.W_Q(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        K = self.W_K(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        V = self.W_V(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        A = (Q @ K.transpose(-1, -2) / (self.d_h ** 0.5)).softmax(-1)
        out = (A @ V) * head_mask.view(1, -1, 1, 1)
        out = out.transpose(1, 2).contiguous().view(B, n, d)
        return self.W_O(out)
    model.mha.forward = types.MethodType(patched, model.mha)
    with torch.no_grad():
        seqs, tgts = make_batch(2000)
        flag = seqs[:, -1]
        preds = model(seqs).argmax(-1)
        accA = (preds[flag == VOCAB['flA']] == tgts[flag == VOCAB['flA']]).float().mean().item()
        accB = (preds[flag == VOCAB['flB']] == tgts[flag == VOCAB['flB']]).float().mean().item()
    model.mha.forward = types.MethodType(MHA.forward, model.mha)
    return accA, accB

m = results[2][0]
for label, mask in [('both heads', torch.ones(2)),
                    ('only head 0', torch.tensor([1.0, 0.0])),
                    ('only head 1', torch.tensor([0.0, 1.0]))]:
    accA, accB = eval_with_mask(m, mask)
    print(f'{label:>14}: acc(A) = {accA:.3f}   acc(B) = {accB:.3f}')

**Sub-finding 5.** *Did ablating head 0 selectively hurt one channel and ablating head 1 hurt the other? That's the cleanest possible signature of head specialization. If both ablations hurt both channels equally, the heads are doing redundant work and the chapter's specialization claim doesn't hold here.*

## Finding

*3–6 sentences. Was the $h=1$ ceiling real? Did $h=2$ specialize cleanly per channel, or did training find a more entangled solution? Did per-channel ablation map to specific heads? Updated mental model on when multi-head actually buys you something vs. when one bigger head suffices.*